In [1]:
import importlib
import sys
import torch
import pickle
import os

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')
sys.path.insert(0, '../../../..')
sys.path.insert(0, '../../../../..')
sys.path.insert(0, '../../../../../..')

from model.dropout_uncertainty_enc_dec_LSTM.dropout_uncertainty_model import DropoutUncertaintyEncoderDecoderLSTM

In [2]:
#load model
# CWD is 5 levels below project root, so this path uses ../../../../../ to climb back
# (matches the relative style of the dataset path below).
file_path_model = '../../../../../src/interpretability/improved_pipeline/henryk/helpdesk/improved/Training/pkl/Helpdesk_full_grad_norm_improved_henryk-p1oversampling.pkl'
model = DropoutUncertaintyEncoderDecoderLSTM.load(file_path_model, dropout=0.0)

# Load the dataset
# Use the improved-loader test pkl — its vocabulary matches the model checkpoint above.
# (The legacy encoded_data/test_philipp/ pkl has 16 Activity classes; this model knows 12.)
file_path_data_set = '../../../../../src/interpretability/improved_pipeline/henryk/helpdesk/improved/Loader/pkl/helpdesk_all_5_test.pkl'
helpdesk_test_dataset = torch.load(file_path_data_set, weights_only=False)

Data set categories:  ([('Activity', 12, {'Assign seriousness': 1, 'Closed': 2, 'Create SW anomaly': 3, 'EOS': 4, 'Insert ticket': 5, 'Require upgrade': 6, 'Resolve SW anomaly': 7, 'Resolve ticket': 8, 'Schedule intervention': 9, 'Take in charge ticket': 10, 'Wait': 11}), ('Resource', 24, {'EOS': 1, 'Value 1': 2, 'Value 10': 3, 'Value 11': 4, 'Value 12': 5, 'Value 13': 6, 'Value 14': 7, 'Value 15': 8, 'Value 16': 9, 'Value 17': 10, 'Value 18': 11, 'Value 19': 12, 'Value 2': 13, 'Value 20': 14, 'Value 21': 15, 'Value 22': 16, 'Value 3': 17, 'Value 4': 18, 'Value 5': 19, 'Value 6': 20, 'Value 7': 21, 'Value 8': 22, 'Value 9': 23}), ('Variant index', 170, {'1.0': 1, '10.0': 2, '100.0': 3, '101.0': 4, '102.0': 5, '104.0': 6, '108.0': 7, '109.0': 8, '11.0': 9, '110.0': 10, '111.0': 11, '112.0': 12, '113.0': 13, '119.0': 14, '12.0': 15, '122.0': 16, '123.0': 17, '124.0': 18, '127.0': 19, '128.0': 20, '129.0': 21, '13.0': 22, '131.0': 23, '132.0': 24, '133.0': 25, '134.0': 26, '135.0': 27, '1

In [3]:
import evaluation.probabilistic_evaluation
importlib.reload(evaluation.probabilistic_evaluation)
from evaluation.probabilistic_evaluation import ProbabilisticEvaluation

new_eval = ProbabilisticEvaluation(model=model, 
                                   dataset=helpdesk_test_dataset,
                                   concept_name='Activity',
                                   num_processes=16,
                                   #growing_num_values = [],
                                   growing_num_values = ['case_elapsed_time'],
                                   samples_per_case = 1,
                                   sample_argmax = False,
                                   use_variance_cat = True,
                                   use_variance_num = True,
                                   all_cat=['Activity', 'Resource'],
                                   all_num=['case_elapsed_time', 'event_elapsed_time'])

In [4]:

def save_chunk(results, i):
    chunk_number = (i + 1)
    os.makedirs(output_dir, exist_ok=True)
    filename = os.path.join(output_dir, f'results_part_{chunk_number:03d}.pkl')
    with open(filename, 'wb') as f:
        pickle.dump(results, f)
    print(f"Saved {len(results)} results to {filename}")

output_dir = '../../../../../evaluation_results/Helpdesk/improved'

save_every = 1000000

results = {}
for i, (case_name, prefix_len, prefix, predicted_suffixes, suffix, mean_prediction) in enumerate(new_eval.evaluate(random_order=True)):
    assert((case_name, prefix_len) not in results)
    results[(case_name, prefix_len)] = (prefix, suffix, mean_prediction, predicted_suffixes)
    print(prefix_len, len(suffix))
    print(mean_prediction)
    if (i + 1) % save_every == 0:
        save_chunk(results, i)
        results = {}

if len(results):
    save_chunk(results, i)

  0%|          | 0/916 [00:00<?, ?it/s]

1 3
[{'Activity': 'Take in charge ticket', 'Resource': 'Value 2', 'case_elapsed_time': array([749407.86595498]), 'event_elapsed_time': array([480200.44599192])}, {'Activity': 'Resolve ticket', 'Resource': 'Value 2', 'case_elapsed_time': array([1687362.95775007]), 'event_elapsed_time': array([605697.67911633])}, {'Activity': 'Closed', 'Resource': 'Value 5', 'case_elapsed_time': array([3621025.38554362]), 'event_elapsed_time': array([2148203.64321639])}]
2 2
[{'Activity': 'Resolve ticket', 'Resource': 'Value 2', 'case_elapsed_time': array([1601377.96583293]), 'event_elapsed_time': array([529978.43580281])}, {'Activity': 'Closed', 'Resource': 'Value 5', 'case_elapsed_time': array([3628491.64203985]), 'event_elapsed_time': array([2198602.91119033])}]
3 1
[{'Activity': 'Closed', 'Resource': 'Value 5', 'case_elapsed_time': array([3467562.1147049]), 'event_elapsed_time': array([1875141.92279448])}]
1 3
[{'Activity': 'Take in charge ticket', 'Resource': 'Value 2', 'case_elapsed_time': array([5

2 2
[{'Activity': 'Resolve ticket', 'Resource': 'Value 5', 'case_elapsed_time': array([2106827.36812152]), 'event_elapsed_time': array([741284.173762])}, {'Activity': 'Closed', 'Resource': 'Value 5', 'case_elapsed_time': array([3308363.94296777]), 'event_elapsed_time': array([1643412.45035663])}]
3 1
[{'Activity': 'Closed', 'Resource': 'Value 5', 'case_elapsed_time': array([3634046.10145991]), 'event_elapsed_time': array([2117136.27593251])}]
1 3
[{'Activity': 'Take in charge ticket', 'Resource': 'Value 2', 'case_elapsed_time': array([359832.69161734]), 'event_elapsed_time': array([359583.22767237])}, {'Activity': 'Resolve ticket', 'Resource': 'Value 2', 'case_elapsed_time': array([1163346.54814359]), 'event_elapsed_time': array([358157.42842433])}, {'Activity': 'Closed', 'Resource': 'Value 5', 'case_elapsed_time': array([3650230.91753842]), 'event_elapsed_time': array([2324146.06883243])}]
2 2
[{'Activity': 'Resolve ticket', 'Resource': 'Value 2', 'case_elapsed_time': array([1235972.5

2 2
[{'Activity': 'Resolve ticket', 'Resource': 'Value 5', 'case_elapsed_time': array([1901496.30465958]), 'event_elapsed_time': array([716387.94199004])}, {'Activity': 'Closed', 'Resource': 'Value 5', 'case_elapsed_time': array([3350049.60193901]), 'event_elapsed_time': array([1791805.71817215])}]
3 1
[{'Activity': 'Closed', 'Resource': 'Value 5', 'case_elapsed_time': array([3667634.2039258]), 'event_elapsed_time': array([2171781.44535104])}]
1 5
[{'Activity': 'Take in charge ticket', 'Resource': 'Value 2', 'case_elapsed_time': array([375456.32489162]), 'event_elapsed_time': array([362039.93903714])}, {'Activity': 'Take in charge ticket', 'Resource': 'Value 2', 'case_elapsed_time': array([634868.04073691]), 'event_elapsed_time': array([259262.23388469])}, {'Activity': 'Resolve ticket', 'Resource': 'Value 2', 'case_elapsed_time': array([923170.75713799]), 'event_elapsed_time': array([369063.70486616])}, {'Activity': 'Resolve ticket', 'Resource': 'Value 5', 'case_elapsed_time': array([2

4 5
[{'Activity': 'Take in charge ticket', 'Resource': 'Value 2', 'case_elapsed_time': array([1037101.07139094]), 'event_elapsed_time': array([500372.87004002])}, {'Activity': 'Resolve ticket', 'Resource': 'Value 2', 'case_elapsed_time': array([1114428.65708348]), 'event_elapsed_time': array([397890.01030444])}, {'Activity': 'Closed', 'Resource': 'Value 5', 'case_elapsed_time': array([3199333.16034386]), 'event_elapsed_time': array([2028621.75195728])}]
5 4
[{'Activity': 'Resolve ticket', 'Resource': 'Value 2', 'case_elapsed_time': array([764882.40823197]), 'event_elapsed_time': array([330298.81492925])}, {'Activity': 'Closed', 'Resource': 'Value 5', 'case_elapsed_time': array([2420137.3335514]), 'event_elapsed_time': array([1472177.26504676])}]
6 3
[{'Activity': 'Resolve ticket', 'Resource': 'Value 2', 'case_elapsed_time': array([1264570.84551257]), 'event_elapsed_time': array([542223.35049033])}, {'Activity': 'Closed', 'Resource': 'Value 5', 'case_elapsed_time': array([2711586.850655

...

1 3
[{'Activity': 'Take in charge ticket', 'Resource': 'Value 2', 'case_elapsed_time': array([484322.29530927]), 'event_elapsed_time': array([425759.68391627])}, {'Activity': 'Resolve ticket', 'Resource': 'Value 2', 'case_elapsed_time': array([1442476.07606125]), 'event_elapsed_time': array([503390.50683199])}, {'Activity': 'Closed', 'Resource': 'Value 5', 'case_elapsed_time': array([3653439.63774012]), 'event_elapsed_time': array([2255140.58947478])}]
2 2
[{'Activity': 'Resolve ticket', 'Resource': 'Value 5', 'case_elapsed_time': array([1695459.59488647]), 'event_elapsed_time': array([633432.40109787])}, {'Activity': 'Closed', 'Resource': 'Value 5', 'case_elapsed_time': array([3380262.76229789]), 'event_elapsed_time': array([1903329.29029481])}]
3 1
[{'Activity': 'Closed', 'Resource': 'Value 5', 'case_elapsed_time': array([3727223.77632078]), 'event_elapsed_time': array([2327180.54613139])}]
1 2
[{'Activity': 'Resolve ticket', 'Resource': 'Value 2', 'case_elapsed_time': array([1428007

3 1
[{'Activity': 'Closed', 'Resource': 'Value 5', 'case_elapsed_time': array([3668019.35822263]), 'event_elapsed_time': array([2188323.01057853])}]
1 3
[{'Activity': 'Take in charge ticket', 'Resource': 'Value 2', 'case_elapsed_time': array([481934.81674079]), 'event_elapsed_time': array([410771.99681967])}, {'Activity': 'Resolve ticket', 'Resource': 'Value 2', 'case_elapsed_time': array([1177655.34592073]), 'event_elapsed_time': array([375196.83545016])}, {'Activity': 'Closed', 'Resource': 'Value 5', 'case_elapsed_time': array([3617879.91792536]), 'event_elapsed_time': array([2315936.64028197])}]
2 2
[{'Activity': 'Resolve ticket', 'Resource': 'Value 2', 'case_elapsed_time': array([1423171.09033329]), 'event_elapsed_time': array([463164.99876097])}, {'Activity': 'Closed', 'Resource': 'Value 5', 'case_elapsed_time': array([3611246.48711107]), 'event_elapsed_time': array([2236063.7541932])}]
3 1
[{'Activity': 'Closed', 'Resource': 'Value 5', 'case_elapsed_time': array([3734986.43736498

2 2
[{'Activity': 'Resolve ticket', 'Resource': 'Value 5', 'case_elapsed_time': array([1866507.63554238]), 'event_elapsed_time': array([677409.79206536])}, {'Activity': 'Closed', 'Resource': 'Value 5', 'case_elapsed_time': array([3348204.24459196]), 'event_elapsed_time': array([1823178.30826639])}]
3 1
[{'Activity': 'Closed', 'Resource': 'Value 5', 'case_elapsed_time': array([3726521.37844656]), 'event_elapsed_time': array([2248267.93305977])}]
1 3
[{'Activity': 'Take in charge ticket', 'Resource': 'Value 2', 'case_elapsed_time': array([810671.88501008]), 'event_elapsed_time': array([452313.88635823])}, {'Activity': 'Resolve ticket', 'Resource': 'Value 2', 'case_elapsed_time': array([1459835.17144799]), 'event_elapsed_time': array([456276.13904735])}, {'Activity': 'Closed', 'Resource': 'Value 5', 'case_elapsed_time': array([3605635.88489408]), 'event_elapsed_time': array([2199807.14398433])}]
2 2
[{'Activity': 'Resolve ticket', 'Resource': 'Value 2', 'case_elapsed_time': array([1620914

1 3
[{'Activity': 'Take in charge ticket', 'Resource': 'Value 2', 'case_elapsed_time': array([373319.58888017]), 'event_elapsed_time': array([374308.02080048])}, {'Activity': 'Take in charge ticket', 'Resource': 'Value 2', 'case_elapsed_time': array([601983.43938798]), 'event_elapsed_time': array([245762.51936734])}, {'Activity': 'Resolve ticket', 'Resource': 'Value 2', 'case_elapsed_time': array([874320.3180705]), 'event_elapsed_time': array([356264.82852864])}, {'Activity': 'Resolve ticket', 'Resource': 'Value 2', 'case_elapsed_time': array([2032410.64241522]), 'event_elapsed_time': array([956111.53197614])}, {'Activity': 'Closed', 'Resource': 'Value 5', 'case_elapsed_time': array([3127758.57179513]), 'event_elapsed_time': array([1617937.86095698])}]
2 2
[{'Activity': 'Take in charge ticket', 'Resource': 'Value 2', 'case_elapsed_time': array([585171.00077613]), 'event_elapsed_time': array([337653.06432912])}, {'Activity': 'Resolve ticket', 'Resource': 'Value 2', 'case_elapsed_time': 

Saved 3356 results to ../../../../../evaluation_results/Helpdesk/improved/results_part_3356.pkl
